## 用其他前沿模型的 API（不只 OpenAI）

## 练习目标（理念）

本笔记本做一件事：给 YouTube 视频写**仅基于字幕**的摘要。

- **输入**：一个 YouTube 链接
- **中间步骤**：用 `youtube-transcript-api` 拉取字幕文本
- **输出**：按固定格式的摘要（概述 / 要点 / 人物机构 / 无法核实的说法）
- **后端对比**：同一套提示词，分别走 **Google Gemini**（OpenAI 兼容接口）与本地 **Ollama**（`llama3.2`、`deepseek-r1:1.5b`）

## 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `ai.chat.completions.create(...)` |
| `messages`（system / user） | system 定摘要规则，user 放字幕全文 |
| OpenAI 兼容 `base_url` | Gemini 与 Ollama 都复用 `OpenAI` 客户端，只换地址 |
| 本地开源模型 | Ollama 的 `llama3.2`、`deepseek-r1:1.5b` |
| 工具链组合 | URL → video id → 字幕 → LLM 摘要 |

## 怎么跑

1. 从上到下依次运行单元格（Shift+Enter）
2. `.env` 里准备 `GOOGLE_API_KEY`（Gemini 路径需要）
3. 本地路径需先安装并启动 Ollama，再 `ollama pull` 对应模型
4. 可改 `video_url` 再分别跑 Gemini / Llama / DeepSeek 做对比


## 为什么还要试 Gemini / Ollama？

帮助我们**少花 OpenAI API 的钱**：同一套摘要流程可以接到别的 frontier 模型或本地模型上。

- **Gemini**：走 Google 的 OpenAI 兼容端点，仍用熟悉的 `OpenAI` SDK
- **Ollama**：完全本地，适合反复试验提示词与对比模型风格


In [86]:
# ========== 依赖：安装拉取 YouTube 字幕用的库 ==========

# -q：安静安装；youtube-transcript-api：按 video id 取字幕（transcript）文本
!pip install -q youtube-transcript-api


In [87]:
# ========== 导入：后面摘要流水线要用的工具箱 ==========

# os：读环境变量（Environment Variables），例如 GOOGLE_API_KEY
import os
# load_dotenv：把 .env 里的密钥读进进程环境，避免把密钥写进笔记本
from dotenv import load_dotenv
# Markdown + display：在 Jupyter 里把模型返回的 Markdown 漂亮展示出来
from IPython.display import Markdown, display
# OpenAI 客户端类：这里不仅给 OpenAI 用，也给 Gemini / Ollama 的兼容端点用
from openai import OpenAI
# urlparse / parse_qs：从 YouTube URL 里解析出 video id（路径或查询参数 v=）
from urllib.parse import urlparse, parse_qs
# YouTubeTranscriptApi：官方社区常用的字幕拉取入口
from youtube_transcript_api import YouTubeTranscriptApi


In [88]:
# ========== 环境变量 + Gemini 兼容端点地址 ==========

# override=True：.env 里的值覆盖进程里已有同名环境变量（以文件为准）
load_dotenv(override=True)
# 从环境读取 Google API Key；后面传给 OpenAI 兼容客户端
api_key = os.getenv("GOOGLE_API_KEY")
# Gemini 的 OpenAI 兼容 base_url（路径末尾带 /openai/）；model 名稍后指定
gemini_base = "https://generativelanguage.googleapis.com/v1beta/openai/"


In [89]:
# ========== 创建 Gemini 客户端（OpenAI 兼容） ==========

# 同一个 OpenAI SDK：api_key 用 Google 的，base_url 指到 Gemini 兼容接口
gemini = OpenAI(api_key=api_key, base_url=gemini_base)


In [136]:
# ========== system prompt：只根据字幕摘要，不编造事实 ==========

# 三引号字符串整体发给模型当 system；内容保持英文（改译会改变模型行为）
# 理念：约束「只信字幕」、噪声时标 [unclear]、输出固定小节
system_prompt = """
You are a transcript-based YouTube summarizer.

Your job:
- Summarize ONLY from the transcript text provided by the user.
- Work for any topic (news, music, podcasts, tutorials, interviews, politics, etc.).

Rules:
1) Do not invent facts, names, dates, or speaker identities.
2) If something is unclear or likely mistranscribed, label it as [unclear].
3) If the transcript is noisy/incomplete, say what is uncertain.
4) Keep the summary concise and neutral.
5) Preserve important proper nouns exactly as they appear in transcript.

Output format:
- (1-2 sentences)
- Key points (4-8 bullets)
- People/organizations mentioned
- Claims that could not be verified from transcript
"""


#### 写一个函数：从 YouTube 链接抽出 video id，再拉字幕并摘要

下面先实现 `extract_video_id`（解析链接），再实现 `summarize_youtube_video`（拉字幕 + 调模型）。


In [132]:
# ========== 从 URL 解析 YouTube video id ==========

def extract_video_id(url: str) -> str:
    # 解析 URL：得到 netloc（域名）与 path / query
    p = urlparse(url)
    # 短链 youtu.be/<id>：id 就在 path 里（去掉前导 /）
    if p.netloc in ("youtu.be", "www.youtu.be"):
        return p.path.lstrip("/")
    # 标准 youtube.com/watch?v=<id>：从查询参数 v 取值
    if "youtube.com" in p.netloc:
        return parse_qs(p.query).get("v", [None])[0]
    # 认不出的链接：返回 None，上层再报 Invalid YouTube URL
    return None


In [173]:
# ========== 核心：拉字幕 → 选模型 → Chat Completions 摘要 ==========

# 两种常见失败：字幕被关闭 / 找不到可用字幕轨
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound

def summarize_youtube_video(ai, video_url, model_=None):
    # 先从链接拿到 video id；拿不到就直接失败，避免盲调 API
    video_id = extract_video_id(video_url)
    if not video_id:
        # 错误文案保持英文（原逻辑依赖的字符串，不翻译）
        raise ValueError("Invalid YouTube URL")
    # 创建字幕 API 客户端
    ytt = YouTubeTranscriptApi()
    try:
        # languages=['en']：优先英文字幕；fetch 返回带 .text 字段的片段列表
        transcript = ytt.fetch(video_id, languages=['en'])
    except (TranscriptsDisabled, NoTranscriptFound):
        # 没有字幕时明确失败，而不是拿空文本去骗模型
        raise ValueError("Transcript not available for this video")
    # 把各片段的 text 拼成一整段，作为 user 消息正文
    transcript_text = " ".join(x.text for x in transcript)
    # 按传入的客户端对象（identity）分支选择 model 字符串
    if ai is gemini:
        # Gemini 路径：固定用 gemini-2.5-flash（model id 保持英文原样）
        m = "gemini-2.5-flash"
    elif ai is ollama:
        # Ollama 路径：用 model_ 短名映射到本机已 pull 的模型全名
        if model_ == "deepseek":
            m = "deepseek-r1:1.5b"
        elif model_ == "llama3.2":
            m = "llama3.2"
        elif not model_:
            # 忘了传 model_ 时给出提示，避免静默用错模型
            raise ValueError("For Ollama, please specify a model (e.g. 'deepseek')")
    else:
        # 既不是 gemini 也不是 ollama：不支持
        raise ValueError("Unsupported model")
    # 统一 Chat Completions：system=摘要规则，user=字幕全文
    response = ai.chat.completions.create(
        model = m,
        messages=[
            {"role": "system", "content": system_prompt},
            # user prompt 模板保持英文；字幕原文拼在后面
            {"role": "user", "content": f"Summarize this transcript:\n\n {transcript_text}"}
        ]
    )
    # 打印实际用到的 model id，方便对比多后端时核对
    print(f"Model used: {m}")
    # 取第一条 choice 的 message.content 作为摘要文本返回
    return response.choices[0].message.content


In [157]:
# ========== 展示封装：摘要后再用 Markdown 渲染 ==========

def summarize_video(ai, video_url, model_=None):
    # 调用核心函数拿到纯文本摘要（可能已含 Markdown 列表）
    summary = summarize_youtube_video(ai, video_url, model_)
    # 先显示固定小标题
    display(Markdown(f"### Summary"))
    # 再把模型输出当 Markdown 渲染（要点列表会更好看）
    display(Markdown(summary))


In [159]:
# ========== 路径 A：用 Gemini 摘要指定视频 ==========

# 可改成你要摘要的任意 YouTube 链接（需有英文字幕）
video_url = "https://www.youtube.com/watch?v=c4pQVTVlaKc"

# ai=gemini：走上面创建的 Google 兼容客户端；model_ 在函数内固定为 gemini-2.5-flash
summarize_video(gemini, video_url)


Model used: gemini-2.5-flash


### Summary

The speaker details a series of aggressive actions taken against drug cartels and the government of Venezuela, including designating cartels as foreign terrorist organizations and fentanyl as a weapon of mass destruction. A new military campaign is credited with stopping record amounts of drugs, eliminating a major cartel kingpin, and culminating in the defeat and capture of Venezuelan dictator Nicholas Maduro. The speaker also notes cooperation with Deli Rodriguez, described as the new president of Venezuela, to foster economic gains and hope for the country.

Key points:
*   Large parts of Mexico are controlled by murderous drug cartels, which have been designated as foreign terrorist organizations.
*   Illicit fentanyl has been declared a weapon of mass destruction.
*   A new military campaign has reportedly stopped record amounts of drugs, with sea routes virtually completely blocked.
*   One of the most sinister cartel kingpins has been taken down.
*   America's armed forces defeated an [unclear] enemy, ending the reign of Nicholas Maduro, who was brought to face American justice.
*   This is described as a colossal victory for US security and a new beginning for Venezuela.
*   The US is working with the "new president of Venezuela," Deli Rodriguez, for economic gains.
*   Maduro's heavily protected military fortress, guarded by thousands of soldiers and Russian and Chinese military technology, was swiftly descended upon.

People/organizations mentioned:
*   Nicholas Maduro
*   Deli Rodriguez (new president of Venezuela)
*   Drug cartels
*   America's armed forces
*   Russian military technology
*   Chinese military technology

Claims that could not be verified from transcript:
*   "stopped record amounts of drugs coming into our country and virtually stopped it completely coming in by water or sea."
*   "America's armed forces overwhelmed all defenses and utterly defeated a enemy. good fighters"
*   "This was an absolutely colossal victory for the security of the United States"
*   "This was a major military installation protected by thousands of soldiers and guarded by Russian and Chinese military technology."

### 接下来试 Ollama（本地）

同一条 `video_url`、同一套 `system_prompt`，只把客户端换成指向本机的 OpenAI 兼容端点。


（作者机器上 Ollama 的 3.3 / 4 系列太大跑不动。）

这里改用 **llama3.2**：体积更小，但对摘要任务通常够用。先 `pull`、确认版本与已安装列表，再创建客户端。


In [46]:
# ========== 拉取本地模型权重：llama3.2 ==========

# ollama pull：从注册表下载模型到本机；之后才能用 model="llama3.2" 调用
!ollama pull llama3.2


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 2.3 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 7.5 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  11 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  14 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  20 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  21 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  26 MB/2.0 GB                

In [47]:
# ========== 确认 Ollama CLI 是否可用 ==========

# 打印版本号：能跑通说明命令行工具已安装
!ollama --version


ollama version is 0.17.0


In [48]:
# ========== 列出本机已安装的模型 ==========

# 确认列表里已有 llama3.2（以及后面要用的 deepseek 变体）
!ollama list


NAME               ID              SIZE      MODIFIED       
llama3.2:latest    a80c4f17acd5    2.0 GB    57 seconds ago    
phi3:latest        4f2222927938    2.2 GB    4 days ago        
gemma3:270m        e7d36fb2c3b3    291 MB    4 days ago        


In [51]:
# ========== 探测本机 Ollama HTTP 服务是否在听 ==========

# 注释说明：检查 Ollama 是否真的在跑（不只是 CLI 装好了）
# requests.get 打默认端口 11434；有响应内容通常表示服务已起来
import requests
requests.get("http://localhost:11434").content


b'Ollama is running'

In [52]:
# ========== 创建 Ollama 的 OpenAI 兼容客户端 ==========

# base_url 指向本地 /v1；api_key 对 Ollama 多为占位字符串 "ollama"（本地通常不校验）
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")


In [163]:
# ========== 路径 B：用本地 llama3.2 摘要同一视频 ==========

# 复用同一 video_url；model_="llama3.2" 会映射到 Ollama 模型名 llama3.2
video_url = "https://www.youtube.com/watch?v=c4pQVTVlaKc"

summarize_video(ollama, video_url, "llama3.2")


Model used: llama3.2


### Summary

Here is a summary of the transcript:

The president stated that their new military campaign has stopped record amounts of drugs from entering the country, almost completely halted smuggling by water or sea, and successfully taken down a powerful cartel kingpin, Nicholas Maduro, who was overthrown and brought to face American justice. The success of this operation is seen as a colossal victory for US security and opens up a new beginning for Venezuela.

Key points:
• Large parts of Mexico have been controlled by murderous drug cartels.
• The president declared illicit fentinol a weapon of mass destruction.
• A military campaign has stopped record amounts of drugs from entering the country.
• A powerful cartel kingpin, Nicholas Maduro, was taken down and overthrown.
• Maduro's regime was described as having a "heavily protected" military fortress.

People/organizations mentioned:
- President (implied)
- Deli Rodriguez, new president of Venezuela
- Mexican territory
- Russia
- China

Claims that could not be verified from transcript:
- [unclear] definition or existence of the chemical compound "fentinol"

### 再试 DeepSeek（仍走 Ollama）

下面先腾出磁盘（可选删除更大的 `deepseek-r1:7b`），再拉取更小的 `deepseek-r1:1.5b`，最后用短名 `"deepseek"` 调用同一套 `summarize_video`。


In [170]:
# ==========（可选）删除本机更大的 DeepSeek 权重以腾空间 ==========

# rm：从本机 Ollama 库移除 deepseek-r1:7b；不影响后面要用的 1.5b
!ollama rm deepseek-r1:7b


deleted 'deepseek-r1:7b'


⠙ 


In [172]:
# ========== 拉取较小的 DeepSeek R1 蒸馏版 ==========

# 1.5b：体量更小，适合本机试跑；model id 必须与 summarize 里的映射一致
!ollama pull deepseek-r1:1.5b


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠸ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏  16 KB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  ▏ 6.5 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  ▏  14 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   2% ▕                  ▏  18 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   2% ▕                  ▏  25 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   3% ▕                  ▏  32 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   3% ▕                  ▏  38 MB/1.1

In [179]:
# ========== 路径 C：用 DeepSeek（Ollama）摘要同一视频 ==========

# model_="deepseek" → 函数内映射为 deepseek-r1:1.5b；video_url 沿用上一格赋值
summarize_video(ollama, video_url, "deepseek")


Model used: deepseek-r1:1.5b


### Summary

(1-2 sentences)  
The video discusses how large territories controlled by drug cartels used fentinol as a weapon to stop illegal drug flow and take down key figures, ultimately defeating Donald Maduro in the US. Key people mentioned include drug cartel members and the use of fentinol.  

People/organizations involved: Drug cartels, military forces. Claims not verifiable due to unknown individuals and context.  
Unclear aspects: [large state terms translated as "territories," other variables possibly implied.]